# Week 6 Exercise - The Price is Right (AdnanGobeljic)

This Week 6 contribution follows the course progression:
- load `ed-donner/items_lite`
- evaluate a constant baseline
- train a simple text + metadata regression model
- optionally compare with a zero-shot LLM pricer

Run from repo root or from `week6/`.

In [1]:
import os
import re
import sys
from pathlib import Path

import numpy as np
import scipy.sparse as sp
from dotenv import load_dotenv
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
print("Starting week 6 exercise...")

def resolve_week6_root() -> Path:
    current = Path.cwd().resolve()
    for base in [current, *current.parents]:
        candidate = base / "week6"
        if (candidate / "pricer" / "items.py").exists():
            return candidate
        if base.name == "week6" and (base / "pricer" / "items.py").exists():
            return base
    raise FileNotFoundError("Could not find week6/pricer from current working directory")
print("Resolving week 6 root...")

week6_root = resolve_week6_root()
print("Inserting week 6 root to sys.path...")
sys.path.insert(0, str(week6_root))
print("Week 6 root inserted to sys.path.")

from pricer.items import Item
from pricer.evaluator import evaluate

load_dotenv(override=True)
print("Loading environment variables...")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print("Environment variables loaded.")

print(f"Week 6 root: {week6_root}")
print("OpenAI key configured:" , bool(OPENAI_API_KEY))

Starting week 6 exercise...
Resolving week 6 root...
Inserting week 6 root to sys.path...
Week 6 root inserted to sys.path.
Loading environment variables...
Environment variables loaded.
Week 6 root: D:\1playground\llm_engineering\week6
OpenAI key configured: True


In [12]:
# Load curated course dataset from Hugging Face.

DATASET = "ed-donner/items_full"
train, val, test = Item.from_hub(DATASET)

print(f"Train: {len(train):,} | Validation: {len(val):,} | Test: {len(test):,}")
print("Example item:", train[0])

Train: 800,000 | Validation: 10,000 | Test: 10,000
Example item: title='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)' category='Tools_and_Home_Improvement' price=64.3 full=None weight=1.5 summary='Title: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single‑piece oil‑rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4" minimum center‑to‑center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation.' prompt=None id=None


In [13]:
# Baseline model: always predict average training price.

MEAN_PRICE = float(np.mean([item.price for item in train]))
print(f"Mean training price: ${MEAN_PRICE:.2f}")


def constant_pricer(item: Item) -> float:
    return MEAN_PRICE

Mean training price: $140.57


In [14]:
# Evaluate baseline.

evaluate(constant_pricer, test, size=120)

  0%|          | 0/120 [00:00<?, ?it/s]

$78 $25 $86 $71 $111 $89 $4 $75 $105 $189 $572 $238 $121 $86 $61 $108 $61 $91 $70 $22 $7 $17 $56 $34 $191 $312 $354 $121 $42 $61 $121 $81 $19 $60 $25 $678 $81 $85 $73 $103 $59 $61 $106 $114 $79 $116 $123 $109 $5 $61 $105 $11 $334 $21 $87 $6 $134 $101 $62 $129 $95 $63 $50 $31 $488 $51 $99 $304 $16 $65 $109 $124 $139 $122 $91 $105 $16 $131 $124 $122 $21 $129 $111 $42 $114 $81 $42 $165 $21 $95 $119 $46 $121 $106 $132 $88 $107 $17 $129 $434 $41 $24 $104 $2 $108 $23 $116 $259 $110 $158 $81 $174 $110 $12 $55 $29 $116 $121 $85 $38 

In [15]:
# Train a stronger local model (TF-IDF + metadata + Ridge).

TRAIN_LIMIT = 12000
train_subset = train[:TRAIN_LIMIT]


def item_to_text(item: Item) -> str:
    parts = [item.title or "", item.summary or "", item.category or ""]
    return "\n".join(parts)


def item_to_numeric(item: Item) -> list[float]:
    title = item.title or ""
    summary = item.summary or ""
    return [
        float(item.weight or 0.0),
        float(len(title)),
        float(len(summary)),
    ]


vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
)
X_text = vectorizer.fit_transform([item_to_text(item) for item in train_subset])
X_num = sp.csr_matrix([item_to_numeric(item) for item in train_subset])
X_train = sp.hstack([X_text, X_num], format="csr")
y_train = np.array([item.price for item in train_subset], dtype=float)

ml_model = Ridge(alpha=1.0, random_state=42)
ml_model.fit(X_train, y_train)

print(f"Trained Ridge model on {len(train_subset):,} samples")

Trained Ridge model on 12,000 samples


In [16]:
def ml_pricer(item: Item) -> float:
    x_text = vectorizer.transform([item_to_text(item)])
    x_num = sp.csr_matrix([item_to_numeric(item)])
    x = sp.hstack([x_text, x_num], format="csr")
    prediction = float(ml_model.predict(x)[0])
    return float(np.clip(prediction, 0.0, 3000.0))


evaluate(ml_pricer, test, size=120)

  0%|          | 0/120 [00:00<?, ?it/s]

$14 $129 $24 $16 $129 $102 $53 $32 $26 $151 $411 $161 $88 $108 $43 $50 $14 $3 $12 $26 $39 $84 $1 $55 $219 $295 $155 $27 $70 $40 $86 $76 $44 $61 $31 $521 $73 $38 $120 $9 $69 $1 $45 $78 $78 $85 $20 $37 $50 $76 $21 $76 $192 $23 $118 $26 $1 $145 $18 $12 $26 $0 $29 $30 $260 $9 $78 $167 $17 $116 $32 $13 $70 $31 $48 $11 $191 $21 $15 $73 $20 $111 $56 $50 $48 $45 $79 $105 $61 $194 $9 $58 $10 $0 $21 $55 $23 $12 $145 $156 $17 $2 $22 $46 $35 $16 $52 $195 $31 $101 $37 $14 $87 $84 $3 $176 $128 $1 $14 $140 

In [28]:
# Optional zero-shot LLM predictor.

from openai import OpenAI

OPENAI_MODEL = "gpt-oss:120b"
openai_client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),
    base_url=os.environ.get("OPENAI_BASE_URL")
) if os.environ.get("OPENAI_API_KEY") else None

SYSTEM_PROMPT = """
You estimate e-commerce prices in USD.
Reply with only one number (no symbols or explanation).
"""


def llm_pricer(item: Item) -> float:
    if not openai_client:
        return ml_pricer(item)

    user_text = (item.summary or item.title or "")
    # response = openai_client.chat.completions.create(
    #     model=OPENAI_MODEL,
    #     temperature=0,
    #     max_tokens=20,
    #     messages=[
    #         {"role": "system", "content": SYSTEM_PROMPT},
    #         {"role": "user", "content": f"Product details:\n{user_text}\n\nPredicted price:"},
    #     ],
    # )
    print("user text:", user_text)
    print("item",item)
    print("item_summary",item.summary)
    print("item_title",item.title)
    print("LLM response received.\n")
    # print(response.choices[0])
    # text = (response.choices[0].message.content or "").strip()
    # print(text)
    match = False  # re.search(r"[-+]?\d*\.?\d+", text)
    if not match:
        # print("No match found in LLM response.")
        return ml_pricer(item)
    # print("Match found, using LLM predicted price.")
    return max(0.0, float(match.group()))
evaluate(llm_pricer, test, size=40)

user text: Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  
Details: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.
item title='Old Blood Noise Excess V2 Distortion Chorus/Delay Pedal' category='Musical_Instruments' price=219.0 full=None weight=2.0 summary='Title: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switchin

  0%|          | 0/40 [00:00<?, ?it/s]

$14 $129 $24 $16 $129 $102 $53 $32 $26 $151 $411 $161 $88 $108 $43 $50 $14 $3 $12 $26 $39 $84 $1 $55 $219 $295 $155 $27 $70 $40 $86 $76 $44 $61 $31 $521 $73 $38 $120 $9 

In [ ]:
# Optional zero-shot LLM predictor.

from openai import OpenAI

OPENAI_MODEL = "gpt-oss:20b"
openai_client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),
    base_url=os.environ.get("OPENAI_BASE_URL")
) if os.environ.get("OPENAI_API_KEY") else None

SYSTEM_PROMPT = """
You estimate e-commerce prices in USD.
Reply with only one number (no symbols or explanation).
"""


def llm_pricer(item: Item) -> float:
    if not openai_client:
        return ml_pricer(item)

    user_text = (item.summary or item.title or "")[:900]
    response = openai_client.chat.completions.create(
        model=OPENAI_MODEL,
        temperature=0,
        max_tokens=20,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Product details:\n{user_text}\n\nPredicted price:"},
        ],
    )

    text = (response.choices[0].message.content or "").strip()
    match = re.search(r"[-+]?\d*\.?\d+", text)
    if not match:
        return ml_pricer(item)

    return max(0.0, float(match.group()))
evaluate(llm_pricer, test, size=40)

  0%|          | 0/40 [00:00<?, ?it/s]

$14 $129 $24 $16 $129 $102 $53 $32 $26 $151 $411 $161 $88 $108 $43 $50 $14 $3 $12 $26 $39 $84 $1 $55 $219 $295 $155 $27 $70 $40 $86 $76 $44 $61 $31 $521 $73 $38 $120 $9 